# Caso H · 01 Arquitectura del chatbot — tools sobre InfluxDB + RAG documental

> _Tutorial · Caso de uso: **H — RAG + Chatbot** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Comprender el patrón **tools (datos numéricos) + RAG (conocimiento general)** y definir el conjunto de herramientas mínimo del chatbot.


## 2. Qué se aprende

- Decisión: pregunta → tool o pregunta → RAG.
- 6 herramientas básicas (`query_influxdb`, `compare_periods`, ...).
- Cómo mockear modelos predictivos para no bloquearse.
- Cómo separar conocimiento factual de conocimiento documental.


## 3. Contexto del caso de uso

Equipo H construye el chatbot integrador: usa modelos B y C/E como tools.


## 4. Relación con CENTINELA+

El chatbot va a producción tras la integración de los modelos en semana 3.


## 5. Relación con Medallion

Consume plata; oro = tools + RAG.


## 6. Datos de entrada

Conceptual.


## 7. Schema CAPTIA esperado

No aplica.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Tabla pregunta → mecanismo.


In [ ]:
mapping = pd.DataFrame(
    [
        ("Dato puntual histórico", "tool: query_influxdb"),
        ("Comparación entre periodos", "tool: compare_periods"),
        ("Predicción meteo / consumo", "tool: get_*_prediction (mock o real)"),
        ("Estado actual del edificio", "tool: get_building_state"),
        ("Detección anomalía HVAC", "tool: check_hvac_anomaly"),
        ("Conocimiento general / normativa", "RAG sobre docs"),
    ],
    columns=["pregunta", "mecanismo"],
)
mapping


## 10. Exploración paso a paso

Diagrama Mermaid.


In [ ]:
from IPython.display import Markdown
Markdown('''```mermaid
flowchart LR
  Q[Pregunta] --> R{Decisión}
  R -- numérica --> T[Tools InfluxDB]
  R -- predicción --> P[Tools mocked → reales]
  R -- documental --> S[RAG ElasticSearch]
  T --> A[Respuesta]
  P --> A
  S --> A
```''')


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Tools en notebooks 02-03; RAG en 04.


## 13. Visualizaciones explicativas

Distribución del golden set por categoría.


In [ ]:
golden = pd.read_csv(ROOT / "notebooks/_data/chatbot_golden_set.csv")
golden["category"].value_counts().plot.bar(color="#3F51B5")
plt.title("Golden set — preguntas por categoría")
plt.tight_layout()


## 14. Validaciones

Tabla cargada.


## 15. Errores comunes

1. Indexar valores numéricos en ElasticSearch (incorrecto: usar tool).
2. Mockear modelos sin firma estable (cambiar firma rompe integraciones).
3. No registrar la trazabilidad de qué tool eligió el agente.


## 16. Ejercicios propuestos

1. Añade una categoría 'control' y discute si requiere tool nueva.
2. Diseña una política de fallback si la tool falla.
3. Discute si los predictores deberían ir en MLflow Server.


## 17. Cómo se reutiliza con datos reales

Cambiar `INFLUXDB_*` y endpoints de modelos. La arquitectura es estable.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `08_case_H_rag_chatbot/02_tools_influxdb.ipynb`.
- Documento web del caso: `docs/use-cases/case-h-rag-chatbot.md`.
